In [ ]:
import os
from getpass import getpass

os.environ["GH_TOKEN"] = getpass("Paste GitHub token: ")
os.environ["HF_TOKEN"] = getpass("Paste Hugging Face token: ")

In [ ]:
!pip install -U huggingface_hub
!hf auth login --token $HF_TOKEN

In [ ]:
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://${GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
%pip install .

In [ ]:
!git pull

In [ ]:
# Download processed data (train/eval split)
!gdown 1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
!mkdir -p data/processed
!unzip -o data_processed.zip -d data/processed
!rm -rf data/processed/__MACOSX data_processed.zip
!ls data/processed/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/drive/MyDrive/vnlegal-rag-v2/negatives/* data/processed/

In [ ]:
!python scripts/update_batch_size.py --vram auto

## Model Selection (n=10, BCE)

In [ ]:
# PhoRanker (n=10, BCE)
!python scripts/train_cross.py --config configs/train/cross_encoder_phranker.yaml

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="models/CrossEncoder/PhoRanker/best",
    repo_id="phatvucoder/PhoRanker-legal-vn",
    repo_type="model"
)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="models/CrossEncoder/PhoRanker/best",
    repo_id="phatvucoder/PhoRanker-legal-vn-v2",
    repo_type="model"
)

In [ ]:
!cat results/train/cross-encoder-phranker-n10-bce.json

## Optimize n + Loss

In [ ]:
import yaml

# >>> SET AFTER PHASE 1 <<<
BEST_CONFIG = "configs/train/cross_encoder_phranker.yaml"  # or cross_encoder_gte.yaml

with open(BEST_CONFIG) as f:
    base = yaml.safe_load(f)

base

In [ ]:
from vnlegal_rag_v2.training import CrossEncoderTrainer

for n in [5, 10, 15, 20]:
    for loss in ["bce", "ranknet"]:
        name = f"cross_{base['name'].split('_n')[0]}_n{n}_{loss}"
        out_dir = f"models/CrossEncoder/{base['model_name'].split('/')[-1]}_n{n}_{loss}"

        print(f"\n{'='*60}")
        print(f"Training: {name}")
        print(f"{'='*60}")

        t = CrossEncoderTrainer(
            model_name_or_path=base["model_name"],
            max_length=base["max_length"],
            trust_remote_code=base["trust_remote_code"],
        )
        t.train(
            data_path=base["data_path"],
            neg_path=f"data/processed/{n}_moderate_neg.csv",
            output_dir=out_dir,
            num_epochs=base["num_epochs"],
            batch_size=base["batch_size"],
            learning_rate=base["learning_rate"],
            segmentation=base.get("segmentation"),
            loss_type=loss,
            seed=base["seed"],
        )
        print(f"✓ {name} → {out_dir}/best")